# Lab 2 — Su primer motor de búsqueda
## TF-IDF + similitud coseno, implementados desde cero

---

| Campo | Detalle |
|---|---|
| **Alumno** | Carlos Rafael Ramos Molina |
| **Matrícula** | 231232 |
| **Profesor** | Ramses Camas Nájera |
| **Grupo** | 9-C |
| **Carrera** | Ingeniería de Software |
| **Materia** | Minería de Datos |
| **Unidad** | Unidad 2 — Del texto al significado |
| **Institución** | Universidad Politécnica de Chiapas (UPCh) |
| **Período** | 2026A |

---

In [10]:
import json, math, operator, re, unicodedata
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)

documentos = [d['tokens'] for d in corpus]
print(f'{len(corpus)} documentos. Ejemplo {corpus[0]["id"]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


In [3]:
import re, unicodedata
from collections import Counter
import spacy

nlp = spacy.load('es_core_news_sm')

RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')

CONSERVAR = {'no', 'sin', 'nunca', 'jamás'}
MIS_STOPWORDS = set(nlp.Defaults.stop_words) - CONSERVAR

def normalizar(texto, quitar_acentos=False):
    texto = texto.lower()
    texto = unicodedata.normalize('NFC', texto)
    texto = RE_HTML.sub(' ', texto)
    texto = RE_URL.sub(' ', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) not in ('So', 'Cs') or c.isascii())
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
        texto = unicodedata.normalize('NFC', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def preprocesar(texto):
    texto_norm = normalizar(texto, quitar_acentos=False)
    doc = nlp(texto_norm)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_punct
        and not token.is_space
        and token.lemma_.lower() not in MIS_STOPWORDS
        and len(token.lemma_) > 1
        and not token.like_num
    ]
    return tokens

print('preprocesar lista. Ejemplo:', preprocesar('La sequía afecta los cultivos')[:6])

preprocesar lista. Ejemplo: ['sequía', 'afectar', 'cultivo']


## 0 · Cargar el corpus procesado del Lab 1
El archivo `corpus_procesado.json` fue generado en el Lab 1 con nuestro pipeline de preprocesamiento. Cada documento ya está tokenizado y lematizado.

In [4]:
import spacy
nlp = spacy.load('es_core_news_sm')

RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')

CONSERVAR = {'no', 'sin', 'nunca', 'jamás'}
MIS_STOPWORDS = set(nlp.Defaults.stop_words) - CONSERVAR

def normalizar(texto, quitar_acentos=False):
    texto = texto.lower()
    texto = unicodedata.normalize('NFC', texto)
    texto = RE_HTML.sub(' ', texto)
    texto = RE_URL.sub(' ', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) not in ('So', 'Cs') or c.isascii())
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
        texto = unicodedata.normalize('NFC', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def preprocesar(texto):
    """Pipeline definitivo: texto crudo -> lista de términos limpios.
    Idéntica a la del Lab 1."""
    texto_norm = normalizar(texto, quitar_acentos=False)
    doc = nlp(texto_norm)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_punct
        and not token.is_space
        and token.lemma_.lower() not in MIS_STOPWORDS
        and len(token.lemma_) > 1
        and not token.like_num
    ]
    return tokens

# Verificación
print('preprocesar lista. Ejemplo:', preprocesar('La sequía afecta los cultivos')[:6])

preprocesar lista. Ejemplo: ['sequía', 'afectar', 'cultivo']


## 1 · Indexar — TF, IDF y TF-IDF desde cero

- **TF** (frecuencia relativa): mide qué tan importante es un término dentro de un documento. Se calcula como `f(t,d) / |d|`.
- **IDF** (frecuencia inversa de documento): mide qué tan raro es un término en todo el corpus. Se calcula como `ln(N / df(t))`. Se cuentan documentos, no apariciones.
- **TF-IDF**: producto de ambos. Un término pesa mucho si aparece frecuentemente en un documento pero es raro en el corpus general.

In [11]:
def tf(doc):
    n = len(doc)
    if n == 0:
        return {}
    return {t: f / n for t, f in Counter(doc).items()}

def idf(corpus):
    N = len(corpus)
    df = Counter(t for doc in corpus for t in set(doc))
    return {t: math.log(N / df[t]) for t in df}

def tfidf(doc, idf_):
    return {t: w * idf_.get(t, 0.0) for t, w in tf(doc).items()}

# Construir el índice
IDF    = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

# Sanity check
top = sorted(INDICE[3].items(), key=operator.itemgetter(1), reverse=True)[:5]
print('Términos top de', corpus[3]['id'], '->', [(t, round(w, 3)) for t, w in top])

Términos top de d04 -> [('gravemente', 0.165), ('cultivo', 0.165), ('maiz', 0.165), ('frijol', 0.165), ('fronterizo', 0.165)]


## 2 · Procesar la consulta

La consulta pasa por exactamente el mismo `preprocesar` que el corpus. Se convierte en vector TF-IDF usando el **mismo IDF** del corpus (no se recalcula con la consulta incluida, eso alteraría los pesos).

In [12]:
def vectorizar_consulta(texto):
    tokens = preprocesar(texto)
    pesos_tf = tf(tokens)
    return {t: w * IDF.get(t, 0.0) for t, w in pesos_tf.items()}

print(vectorizar_consulta('sequia en los cultivos'))

{'sequia': 0.9729550745276566, 'cultivo': 1.3195286648076292}


## 3 · Ranquear — similitud coseno

La similitud coseno mide el ángulo entre dos vectores. Si los vectores apuntan en la misma dirección (comparten términos con pesos similares), el coseno se acerca a 1. Si no comparten ningún término, el coseno es 0.

Se trabaja con vectores dispersos (dicts) para eficiencia: solo se opera sobre los términos que ambos vectores tienen en común.

In [13]:
def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts).
    Maneja el caso de norma 0."""
    comunes = set(v1) & set(v2)
    numerador = sum(v1[t] * v2[t] for t in comunes)
    norma1 = math.sqrt(sum(w * w for w in v1.values()))
    norma2 = math.sqrt(sum(w * w for w in v2.values()))
    if norma1 == 0 or norma2 == 0:
        return 0.0
    return numerador / (norma1 * norma2)

def buscar(consulta, k=5):
    """Devuelve los k documentos más relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(consulta)
    ranking = [
        (coseno(q, INDICE[i]), corpus[i]['id'], corpus[i]['titulo'])
        for i in range(len(corpus))
    ]
    ranking.sort(reverse=True)
    return ranking[:k]

# Prueba
print('Consulta: "sequia y cultivos de maiz"')
for score, id_, titulo in buscar('sequia y cultivos de maiz'):
    print(f'{score:.3f}  {id_}  {titulo}')

Consulta: "sequia y cultivos de maiz"
0.431  d04  Sequia afecta cultivos de maiz
0.083  d02  Crisis hidrica golpea la region
0.000  d14  Estudiantes ganan concurso de robotica
0.000  d13  Restablecen servicio de agua potable
0.000  d12  Feria celebra el cafe y el cacao


## 4 · Rómpanlo

Entender dónde falla el modelo vale más que celebrarlo donde funciona.
El caso de clase: `"problemas de agua"` debería recuperar también `d02` (crisis hídrica),
pero TF-IDF no sabe que *agua* e *hídrico* son sinónimos — son cadenas distintas, coseno = 0.

In [14]:
print('Consulta: "problemas de agua"')
for score, id_, titulo in buscar('problemas de agua'):
    print(f'{score:.3f}  {id_}  {titulo}')

Consulta: "problemas de agua"
0.476  d13  Restablecen servicio de agua potable
0.000  d14  Estudiantes ganan concurso de robotica
0.000  d12  Feria celebra el cafe y el cacao
0.000  d11  Alertan por casos de dengue
0.000  d10  Avanza obra de infraestructura carretera


## 4.a Consulta fallida 1 — Sinonimia

**Consulta:** `"daños por terremoto"`

**Causa:** El corpus usa el término *sismo* (d06), no *terremoto*. Para TF-IDF son símbolos completamente distintos aunque signifiquen lo mismo. El vector de la consulta no comparte ningún término con d06, por lo que el coseno es 0 y el documento relevante queda invisible.

In [15]:
print('Consulta fallida 1: "daños por terremoto"')
for score, id_, titulo in buscar('daños por terremoto'):
    print(f'{score:.3f}  {id_}  {titulo}')

Consulta fallida 1: "daños por terremoto"
0.000  d14  Estudiantes ganan concurso de robotica
0.000  d13  Restablecen servicio de agua potable
0.000  d12  Feria celebra el cafe y el cacao
0.000  d11  Alertan por casos de dengue
0.000  d10  Avanza obra de infraestructura carretera


## 4.a Consulta fallida 2 — Término fuera del vocabulario (OOV)

**Consulta:** `"contaminación del río"`

**Causa:** Ningún documento del corpus contiene los términos *contaminación* ni *río*. El vector de la consulta queda completamente vacío (todos los pesos son 0), por lo que el coseno con cualquier documento es 0 y el buscador no puede rankear nada. Este es el problema de vocabulario fuera de dominio (Out-Of-Vocabulary).

In [16]:
print('Consulta fallida 2: "contaminación del río"')
for score, id_, titulo in buscar('contaminación del río'):
    print(f'{score:.3f}  {id_}  {titulo}')

Consulta fallida 2: "contaminación del río"
0.000  d14  Estudiantes ganan concurso de robotica
0.000  d13  Restablecen servicio de agua potable
0.000  d12  Feria celebra el cafe y el cacao
0.000  d11  Alertan por casos de dengue
0.000  d10  Avanza obra de infraestructura carretera


## 5 · (Opcional) Verificación contra scikit-learn

Se compara el ranking propio contra `TfidfVectorizer` de scikit-learn. Los pesos absolutos difieren porque sklearn usa IDF suavizado y normalización L2, pero el **orden del ranking debe coincidir**, lo que valida nuestra implementación desde cero.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs_txt = [' '.join(t) for t in documentos]
vec = TfidfVectorizer(token_pattern=r'\S+')
X   = vec.fit_transform(docs_txt)
q   = vec.transform([' '.join(preprocesar('sequia y cultivos de maiz'))])
sims = cosine_similarity(q, X)[0]

ref = sorted(zip(sims, [d['id'] for d in corpus]), reverse=True)[:5]
print('sklearn:', [(i, round(float(s), 3)) for s, i in ref])
print('propio :', [(i, round(s, 3)) for s, i, _ in buscar('sequia y cultivos de maiz')])
// Las diferencias se deben a que sklearn normaliza los vectores, mientras que el propio no.

sklearn: [('d04', 0.432), ('d02', 0.107), ('d14', 0.0), ('d13', 0.0), ('d12', 0.0)]
propio : [('d04', 0.431), ('d02', 0.083), ('d14', 0.0), ('d13', 0.0), ('d12', 0.0)]
